## Question 1:

**Input**

| a | b |
|---|---|
| 1 | 5 |
| 1 | 5 |
| 1 | 5 |
| 2 | 6 |
| 1 | 5 |

**Expected Output**

| a | sum_b |
|---|-------|
| 1 | 20    |

In [0]:
data = [
    (1, 5),
    (1, 5),
    (1, 5),
    (2, 6),
    (1, 5)
]

df = spark.createDataFrame(
    data,
    ["a", "b"]
)

df.show()

df.createOrReplaceTempView('table_name')

In [0]:
%sql
select a, sum(b) as sum_b from table_name group by a having sum(b) > 10

In [0]:
from pyspark.sql import functions as F
df = df.groupBy(F.col('a'))\
        .agg(F.sum(F.col('b')).alias('sum_b'))\
        .filter(F.col('sum_b') > 10)


df.display()

In [0]:
import pandas as pd

data = {"a": [1,1,1,2,1], "b":[5,5,5,6,5]}

df = pd.DataFrame(data)

result = (
  df.groupby("a", as_index=False)["b"]
    .sum()
    .rename(columns={"b":"sum_b"})
    .query("sum_b > 10")
)

print(result)

Question 2:


| Brand_1 | Brand_2 | Brand_3 | Winner |
|---------|---------|---------|--------|
| A       | B       | C       | B      |
| B       | C       | E       | E      |
| C       | A       | D       | D      |
| D       | E       | A       | A      |
| F       | B       | C       | F      |



Expected Output


| Brand_Name | Total_Appearances | Wins | Losses |
|------------|-------------------|-----|-------|
| A          | 3                 | 1   | 2     |
| B          | 3                 | 1   | 2     |
| C          | 4                 | 0   | 4     |
| D          | 2                 | 1   | 1     |
| E          | 2                 | 1   | 1     |
| F          | 1                 | 1   | 0     |


In [0]:

data = [
    ("A", "B", "C", "B"),
    ("B", "C", "E", "E"),
    ("C", "A", "D", "D"),
    ("D", "E", "A", "A"),
    ("F", "B", "C", "F")
]

df = spark.createDataFrame(
    data,
    ["Brand_1", "Brand_2", "Brand_3", "Winner"]
)

df.createOrReplaceTempView("table_name")

In [0]:
%sql
with cte1 as
(
select Brand_1 as Brand_Name, case when Brand_1 = Winner then 1 else 0 end as Winner from table_name
union all
select Brand_2 as Brand_Name, case when Brand_2 = Winner then 1 else 0 end as Winner from table_name
union all
select Brand_3 as Brand_Name, case when Brand_3 = Winner then 1 else 0 end as Winner from table_name
)
select Brand_Name, count(Brand_Name) as Total_Appearances, sum(Winner) as Wins, (count(Brand_Name) - sum(Winner)) as Losses from cte1 group by Brand_Name

In [0]:
from pyspark.sql import functions as F
df1 = df.withColumn('Brand_Name', F.col('Brand_1'))\
		.withColumn('Winner', F.when(F.col('Brand_1') == F.col('Winner'), 1).otherwise(0))\
		.unionByName(
			df.withColumn('Brand_Name', F.col('Brand_2'))\
			.withColumn('Winner', F.when(F.col('Brand_2') == F.col('Winner'), 1).otherwise(0))\
			.unionByName(
				df.withColumn('Brand_Name', F.col('Brand_3'))\
				.withColumn('Winner', F.when(F.col('Brand_3') == F.col('Winner'), 1).otherwise(0))
			)
		)\
		.groupBy(F.col('Brand_Name'))\
		.agg(F.count('Brand_Name').alias('Total_Appearances'), F.sum(F.col('Winner')).alias('Winner'), (F.count('Brand_Name') - F.sum('Winner')).alias('Losses'))\
		.select('Brand_Name', 'Total_Appearances', 'Winner', 'Losses')
  
df1.display()

In [0]:
import pandas as pd
import numpy as np

pdf = df.toPandas()

df_b1 = pdf.assign(
    Brand_Name=pdf["Brand_1"], Winner=np.where(pdf["Brand_1"] == pdf["Winner"], 1, 0)
)

df_b2 = pdf.assign(
    Brand_Name=pdf["Brand_2"], Winner=np.where(pdf["Brand_2"] == pdf["Winner"], 1, 0)
)

df_b3 = pdf.assign(
    Brand_Name=pdf["Brand_3"], Winner=np.where(pdf["Brand_3"] == pdf["Winner"], 1, 0)
)

df1 = pd.concat([df_b1, df_b2, df_b3], ignore_index=True)

result = (
    df1.groupby("Brand_Name")
    .agg(Total_Appearances=("Brand_Name", "count"), Winner=("Winner", "sum"))
    .reset_index()
)

result["Losses"] = result["Total_Appearances"] - result["Winner"]

print(result)

## Question 3

**Input Table**

| Team_1 | Team_2 | Winner |
|--------|--------|--------|
| India  | SL     | India  |
| SL     | Aus    | Aus    |
| SA     | Eng    | Eng    |
| Eng    | NZ     | NZ     |
| Aus    | India  | India  |

**Expected Output**

| Team_Name | Matches_played | no_of_wins | no_of_losses |
|-----------|---------------|------------|--------------|
| India     | 2             | 2          | 0            |
| SL        | 2             | 0          | 2            |
| Aus       | 2             | 1          | 1            |
| SA        | 1             | 0          | 1            |
| Eng       | 2             | 1          | 1            |
| NZ        | 1             | 1          | 0            |

In [0]:
#from pyspark.sql import SparkSession

data = [
    ("India", "SL", "India"),
    ("SL", "Aus", "Aus"),
    ("SA", "Eng", "Eng"),
    ("Eng", "NZ", "NZ"),
    ("Aus", "India", "India")
]

columns = ["Team_1", "Team_2", "Winner"]

df = spark.createDataFrame(data, columns)

df.createOrReplaceTempView("table_name")

df.show()

In [0]:
%sql
with cte1 as 
(
select Team_1 as Team_Name, case when Team_1 = Winner then 1 else 0 end as Winner from table_name
union all
select Team_2 as Team_Name, case when Team_2 = Winner then 1 else 0 end as Winner from table_name
)
select Team_Name, count(Team_Name) as Matched_played, sum(Winner) as no_of_wins, (count(Winner) - sum(Winner)) as no_of_losses from cte1 group by Team_Name

In [0]:
from pyspark.sql import functions as F

df1 = df.withColumn('Team_Name', F.col('Team_1'))\
	   .withColumn('Winner', F.when(F.col('Team_1') == F.col('Winner'), 1).otherwise(0))\
	   .unionByName(
		   df.withColumn('Team_Name', F.col('Team_2'))\
		   .withColumn('Winner', F.when(F.col('Team_2') == F.col('Winner'), 1).otherwise(0))
	   )\
	   .groupBy(F.col('Team_Name'))\
	   .agg(F.count('Team_Name').alias('Matched_played'), F.sum('Winner').alias('no_of_wins'))\
	   .select('Team_Name','Matched_played','no_of_wins',(F.col('Matched_played') - F.col('no_of_wins')).alias('no_of_losses'))

df1.display()

In [0]:
import pandas as pd
import numpy as np

pdf = df.toPandas()

df_t1 = pdf.assign(
    Team_Name=pdf["Team_1"], Winner=np.where(pdf["Team_1"] == pdf["Winner"], 1, 0)
)

df_t2 = pdf.assign(
    Team_Name=pdf["Team_2"], Winner=np.where(pdf["Team_2"] == pdf["Winner"], 1, 0)
)

df1 = pd.concat([df_t1, df_t2], ignore_index=True)

result = (
    df1.groupby("Team_Name")
    .agg(Matches_played=("Team_Name", "count"), no_of_wins=("Winner", "sum"))
    .reset_index()
)

result["no_of_losses"] = result["Matches_played"] - result["no_of_wins"]

print(result)

## Question 4:

**Input Table**

| sname | sid | marks |
|-------|-----|-------|
| A     | X   | 75    |
| A     | Y   | 75    |
| A     | Z   | 80    |
| B     | X   | 90    |
| B     | Y   | 91    |
| B     | Z   | 75    |

**Expected Output**

| sname | sum(marks) |
|-------|------------|
| A     | 155        |
| B     | 181        |

In [0]:


data = [
    ("A", "X", 75),
    ("A", "Y", 75),
    ("A", "Z", 80),
    ("B", "X", 90),
    ("B", "Y", 91),
    ("B", "Z", 75)
]

columns = ["sname", "sid", "marks"]

df = spark.createDataFrame(data, columns)

df.createOrReplaceTempView("table_name")

df.show()

In [0]:
%sql    
with cte1 as
(
    select *, row_number() over(partition by sname order by marks desc) as rn  from table_name
)
select sname, sum(marks) from cte1 where rn <= 2 group by sname 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df1 = df.withColumn('rn', F.row_number().over(Window.partitionBy(F.col('sname')).orderBy(F.col('marks').desc())))\
		.filter(F.col('rn') <= 2)\
		.groupBy(F.col('sname'))\
		.agg(F.sum(F.col('marks')).alias('sum_marks'))
  
df1.display()

In [0]:
import pandas as pd
import numpy as np

dfp = df.toPandas()

dfp = dfp.sort_values(
    by=["sname", "marks"],
    ascending = [True, False]
)

dfp['rn'] = dfp.groupby('sname').cumcount()+1

dfp = dfp[dfp['rn'] <= 2]

dfp = dfp.groupby('sname', as_index=False).agg(sum_marks = ('marks', 'sum'))

print(dfp)



## Question 5:

**Input Table**

| id |
|----|
|  2 |
|  5 |
|  6 |
|  6 |
|  7 |
|  8 |
|  8 |

**Output Table**

| max_Number |
|------------|
|          7 |


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    (2,),
    (5,),
    (6,),
    (6,),
    (7,),
    (8,),
    (8,)
]

df = spark.createDataFrame(data, ["id"])

df.createOrReplaceTempView("table_name")

df.show()

In [0]:
%sql
with cte1 as
(
    select id, count(id) as count from table_name group by id
)
select max(id) as max_Number from (select id from cte1 where count <= 1)


In [0]:
from pyspark.sql import functions as F

df = df.groupBy(F.col('id'))\
		.agg(F.count(F.col('id')).alias('count'))\
		.filter(F.col('count') <= 1)\
		.agg(F.max(F.col('id')).alias('max_Number'))
  
df.display()

In [0]:
import pandas as pd

dfp = df.toPandas()
dfp = dfp.groupby("id", as_index=False).agg(count=("id", "count"))
dfp = dfp[dfp["count"] <= 1]
result = pd.DataFrame({"max_Number": [dfp["id"].max()]})
print(result)

## Question 6

**table_a**

| empid | ename | salary |
|-------|-------|--------|
| 1     | AA    | 1000   |
| 2     | BB    | 300    |

**table_b**

| empid | ename | salary |
|-------|-------|--------|
| 2     | BB    | 400    |
| 3     | CC    | 100    |

**expected_output**

| empid | ename | salary |
|-------|-------|--------|
| 1     | AA    | 1000   |
| 2     | BB    | 300    |
| 3     | CC    | 100    |

In [0]:
from pyspark.sql import SparkSession


data_a = [
    (1, "AA", 1000),
    (2, "BB", 300)
]

data_b = [
    (2, "BB", 400),
    (3, "CC", 100)
]

df_a = spark.createDataFrame(data_a, ["empid", "ename", "salary"])
df_b = spark.createDataFrame(data_b, ["empid", "ename", "salary"])

df_a.createOrReplaceTempView("table_name1")
df_b.createOrReplaceTempView("table_name2")

df_a.show()
df_b.show()

In [0]:
%sql
with cte1 as
(
select * from table_name1
union all
select * from table_name2
),
cte2 as
(
select *, row_number() over(partition by ename order by salary asc) as rn from cte1
)
select empid, ename, salary from cte2 where rn = 1

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
df = df_a.unionByName(df_b)\
         .withColumn('rn', F.row_number().over(Window.partitionBy('ename').orderBy(F.col('salary'))))\
         .filter(F.col('rn') == 1)\
          .drop('rn')
  
df.display()

In [0]:
import pandas as pd

dfap = df_a.toPandas()
dfbp = df_b.toPandas()

dfp = pd.concat([dfap, dfbp])
dfp = dfp.sort_values(
    by = ['empid', 'salary'],
    ascending = [True, True]
)
dfp['rn'] = dfp.groupby('ename', as_index = False).cumcount() + 1
dfp = dfp[dfp['rn'] == 1]
dfp = dfp.drop(columns = ['rn'])
print(dfp)
